# 第四章补充 4.2：模型微调

预训练模型（Base Model）在通用语料上学到了丰富的语言知识，但要让它在具体任务上表现好，还需要微调。这一章回答两个问题：

- **为什么不能直接全量微调大模型？** — 显存墙、灾难性遗忘
- **怎么高效微调？** — LoRA、PEFT 生态、指令微调

## 目录

| 章节 | 内容 |
|------|------|
| **4.2.1 全量微调的困境** | 为什么微调 70B 模型不现实 |
| **4.2.2 LoRA：低秩适配** | 核心思想、矩阵分解直觉、参数量对比 |
| **4.2.3 PEFT 生态** | Prompt Tuning、Prefix Tuning、LoRA 变体对比 |
| **4.2.4 指令微调** | SFT 数据构造、chat template、与 RLHF 的关系 |

In [ ]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.unicode_minus'] = False

## 4.2.1 全量微调的困境

### 什么是全量微调（Full Fine-tuning）？

拿到一个预训练模型（如 LLaMA-7B），用特定任务的数据**更新模型所有参数**，使其适配目标任务。

理论上，这是最彻底的迁移学习方式，也是传统 BERT fine-tune 的做法（BERT 只有 110M 参数，更新全量参数是可以接受的）。

---

### 为什么对大模型不现实？

以 LLaMA-7B 为例：

```
┌─────────────────────────────────────────────────────┐
│ 全量微调 LLaMA-7B 的显存需求                          │
├─────────────────┬──────────────┬───────────────────┤
│ 组件            │ 精度         │ 显存估算           │
├─────────────────┼──────────────┼───────────────────┤
│ 模型权重        │ BF16 (2B/参数)│ 7B × 2 = 14 GB    │
│ 梯度           │ FP32 (4B/参数)│ 7B × 4 = 28 GB    │
│ Adam 优化器状态 │ FP32 × 2      │ 7B × 8 = 56 GB    │
│ 激活值（batch） │ 与序列长度相关│ ~10-20 GB         │
├─────────────────┴──────────────┴───────────────────┤
│ 合计：约 100-120 GB                                  │
└─────────────────────────────────────────────────────┘
```

**Adam 优化器为什么这么占内存？**

Adam 需要维护两个一阶和二阶矩统计量 $(m_t, v_t)$，每个参数各需要一份 FP32 的存储，合计是参数本身的 **2 倍**大小（即 8 字节/参数）。

**结果**：微调 LLaMA-7B 需要至少 **8 张 A100**（80GB × 8 = 640GB，还要考虑并行通信开销）。对于 70B 模型，这个数字变成 80+ 张 A100，成本极高，绝大多数团队根本负担不起。

---

### 全量微调的另一个问题：灾难性遗忘

就算有钱买足够的 GPU，全量微调还有另一个问题：**灾难性遗忘（Catastrophic Forgetting）**。

模型在特定任务数据上过度微调后，会忘记预训练时学到的通用知识。用医疗问答数据微调后，模型可能在一般问答上明显退化。

---

### 这就是 PEFT 存在的原因

**Parameter-Efficient Fine-Tuning（PEFT，参数高效微调）**：只更新**极少量参数**，冻结大部分预训练权重，同时达到接近全量微调的效果。

```
全量微调：更新 7,000,000,000 个参数  → 需要 100+ GB 显存
LoRA：    更新      3,000,000 个参数  → 需要约 16 GB 显存（↓ 6x）
           （仅占总参数量的 0.04%）
```

下一节介绍最主流的 PEFT 方法——LoRA。

## 4.2.2 LoRA：低秩适配（Low-Rank Adaptation）

### 核心假设

Hu 等人（2021）观察到：微调过程中，权重矩阵的**变化量 $\Delta W$ 具有低秩性**——尽管 $\Delta W$ 的形状和 $W$ 一样大，但它的信息可以用少量方向来表达（即秩很低）。

**直觉类比**：假设你已经掌握了英语，现在要学法语。你不需要重新建立整套语言系统，只需在原有知识上做"小幅调整"（学新词汇、调整语序习惯）。这些调整的"信息量"远比从零学一门语言少得多。

---

### 数学原理

全量微调：更新整个权重矩阵 $W + \Delta W$，其中 $\Delta W \in \mathbb{R}^{d \times k}$

**LoRA 的做法**：不直接学 $\Delta W$，而是把它分解为两个小矩阵的乘积：

$$\Delta W = B \cdot A, \quad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k}, \quad r \ll \min(d, k)$$

推理时的前向传播：

$$h = W x + \Delta W x = W x + B A x$$

$W$ 的权重**冻结不动**，只训练 $A$ 和 $B$。

**初始化策略**：
- $A$ 用高斯随机初始化
- $B$ 初始化为全零，保证训练开始时 $\Delta W = BA = 0$（不破坏预训练权重的输出）

---

### 参数量对比

以 BERT-base 的一个 Attention 层为例（$d = k = 768$）：

$$\text{全量微调：} d \times k = 768 \times 768 = 589{,}824 \text{ 参数}$$

$$\text{LoRA（r=8）：} d \times r + r \times k = 768 \times 8 + 8 \times 768 = 12{,}288 \text{ 参数}$$

$$\text{压缩比} = \frac{589{,}824}{12{,}288} \approx 48×$$

对整个 LLaMA-7B，LoRA（$r=8$）只需训练约 **4-16M 参数**（取决于应用于哪些层），仅为总参数量的 **0.06%-0.2%**。

---

### 关键超参数

**秩 $r$**（最重要）：控制新增参数量和表达能力的平衡。
- $r = 4$：极少参数，适合简单任务微调
- $r = 8$：常用默认值，平衡效果与效率
- $r = 16 \sim 64$：更高表达能力，参数量增加但仍远小于全量微调

**缩放系数 $\alpha$**：实际更新量为 $\frac{\alpha}{r} \cdot BAx$，通常设 $\alpha = r$（等效于不缩放）或 $\alpha = 2r$（放大更新幅度）。

**应用到哪些矩阵**：LoRA 通常应用到 Attention 层的 $W_Q, W_K, W_V, W_O$ 和 FFN 层。只用 $W_Q, W_V$ 也很常见（参数最少）。

---

### QLoRA：4bit 量化 + LoRA

Dettmers 等（2023）将 LoRA 与 4bit 量化结合：

1. 用 NF4 将预训练权重 $W$ 量化为 4bit（显存降至 1/8）
2. 在 4bit 模型上插入 LoRA 适配器，训练时只更新 LoRA 参数（BF16 精度）
3. 梯度通过量化层反传时进行特殊处理

**效果**：在单张 A100（80GB）上可以微调 65B 参数的模型，精度接近全量 BF16 微调。

---

### LoRA 的优缺点

| 优点 | 缺点 |
|------|------|
| 显存极大减少（原全量微调的 1/8~1/4）| 需要选择应用哪些层（超参数） |
| 训练速度大幅提升 | 表达能力弱于全量微调（通常差 0.5-2%）|
| 不破坏预训练权重，可灵活切换任务 | 秩 $r$ 的选择需要一定经验 |
| 训练完成后可合并进原始权重，推理无额外开销 | 对极端分布偏移任务效果有限 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

np.random.seed(42)

fig = plt.figure(figsize=(15, 10))
fig.suptitle('LoRA：低秩适配原理', fontsize=14, fontweight='bold')

# ── 上左图：LoRA 结构示意 ──────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
ax1.set_xlim(0, 10); ax1.set_ylim(0, 10); ax1.axis('off')
ax1.set_title('LoRA 结构：原始路径 + 低秩旁路', fontsize=10, fontweight='bold')

def fbox(ax, x, y, w, h, text, fc='#d0e8ff', ec='#3a7fcc', fs=9, bold=False):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                         boxstyle='round,pad=0.05', facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fs,
            fontweight='bold' if bold else 'normal', wrap=True)

def arr(ax, x0, y0, x1, y1, color='#555'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))

fbox(ax1, 5, 8.5, 3, 0.8, 'Input $x$', '#fff9c4', '#f9a825', bold=True)
fbox(ax1, 5, 6.8, 3, 0.8, '冻结权重 $W$\n（不更新）', '#e8f5e9', '#388e3c')
fbox(ax1, 2.5, 5.0, 2.0, 0.8, 'A（可训练）\nr×k', '#d0e8ff', '#1565c0')
fbox(ax1, 7.5, 5.0, 2.0, 0.8, 'B（可训练）\nd×r', '#d0e8ff', '#1565c0')
fbox(ax1, 5, 3.2, 3, 0.8, '$+$ 加法合并', '#f3e5f5', '#7b1fa2')
fbox(ax1, 5, 1.5, 3, 0.8, 'Output $h$', '#fff9c4', '#f9a825', bold=True)

arr(ax1, 5, 8.1, 5, 7.2)
arr(ax1, 5, 6.4, 5, 3.6)
arr(ax1, 3.5, 8.5, 2.5, 5.4)
arr(ax1, 2.5, 4.6, 7.5, 5.4)
arr(ax1, 7.5, 4.6, 5.5, 3.6)
arr(ax1, 5, 2.8, 5, 1.9)

ax1.text(5.0, 5.0, 'r=8', ha='center', fontsize=8, color='#1565c0', fontweight='bold')
ax1.text(3.1, 8.1, '旁路（ΔW=BA）', ha='center', fontsize=8, color='#1565c0', style='italic')

# ── 上中图：不同秩 r 的参数量 ─────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
d = k = 4096
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
full_params = d * k
lora_params = [2 * d * r for r in ranks]
ratios = [lp / full_params * 100 for lp in lora_params]

ax2.bar(range(len(ranks)), ratios, color='#4a90d9', alpha=0.85, edgecolor='gray')
ax2.set_xticks(range(len(ranks))); ax2.set_xticklabels([f'r={r}' for r in ranks], fontsize=8.5)
ax2.set_ylabel('占全量微调参数比 (%)', fontsize=9)
ax2.set_title(f'LoRA 参数量 vs 秩 r\n（d=k={d}，单层 Q/K/V/O 之一）', fontsize=9)
ax2.set_yscale('log')
for i, (r, rp) in enumerate(zip(ranks, ratios)):
    ax2.text(i, rp * 1.3, f'{rp:.2f}%', ha='center', fontsize=7.5)
ax2.axhline(100, color='red', linestyle='--', linewidth=1, alpha=0.7, label='全量微调（100%）')
ax2.legend(fontsize=8)

# ── 上右图：LLaMA-7B 总训练参数量对比 ─────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
model_params = 7e9
lora_total = {r: 32 * 2 * 2 * 4096 * r for r in [4, 8, 16, 32, 64]}
methods = ['全量微调'] + [f'LoRA\nr={r}' for r in [4, 8, 16, 32, 64]]
params_m = [model_params / 1e6] + [v / 1e6 for v in lora_total.values()]
colors = ['#e07b54'] + ['#4a90d9'] * 5

bars = ax3.bar(methods, params_m, color=colors, edgecolor='gray', width=0.6)
ax3.set_yscale('log')
ax3.set_ylabel('可训练参数量（百万）', fontsize=9)
ax3.set_title('LLaMA-7B：全量 vs LoRA\n可训练参数量对比', fontsize=9)
for bar, p in zip(bars, params_m):
    ax3.text(bar.get_x() + bar.get_width()/2, p * 1.2,
             f'{p:.0f}M', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── 下图：LoRA 显存节省（训练） ───────────────────────────────────────────
ax4 = fig.add_subplot(2, 1, 2)
ax4.set_xlim(0, 12); ax4.set_ylim(0, 10); ax4.axis('off')
ax4.set_title('LLaMA-7B 训练时显存占用分解（估算值）', fontsize=11, fontweight='bold')

categories = ['模型权重\n(BF16)', '梯度\n(FP32)', 'Adam 优化器\n(FP32 × 2)', '激活值\n(batch=4)']
full_vram  = [14, 28, 56, 16]
lora_vram  = [7,   0.1, 0.2, 16]
colors_cat = ['#4a90d9', '#e07b54', '#f0b429', '#6abf69']

bar_h = 0.55
y_full = 7.5
y_lora = 4.5
scale = 0.085

cumx_full = 0
cumx_lora = 0
for i, (cat, fv, lv, col) in enumerate(zip(categories, full_vram, lora_vram, colors_cat)):
    fw = fv * scale
    lw = lv * scale
    rect = plt.Rectangle((cumx_full, y_full - bar_h/2), fw, bar_h, facecolor=col, edgecolor='white', lw=1)
    ax4.add_patch(rect)
    if fv > 5:
        ax4.text(cumx_full + fw/2, y_full, f'{fv}GB', ha='center', va='center', fontsize=8.5, fontweight='bold', color='white')
    cumx_full += fw

    lw_draw = max(lw, 0.02)
    rect2 = plt.Rectangle((cumx_lora, y_lora - bar_h/2), lw_draw, bar_h, facecolor=col, edgecolor='white', lw=1)
    ax4.add_patch(rect2)
    if lv > 5:
        ax4.text(cumx_lora + lw_draw/2, y_lora, f'{lv}GB', ha='center', va='center', fontsize=8.5, fontweight='bold', color='white')
    cumx_lora += lw_draw

total_full = sum(full_vram)
total_lora_est = 7 + 0.1 + 0.2 + 16

ax4.text(-0.2, y_full, '全量微调', ha='right', va='center', fontsize=10, fontweight='bold', color='#333')
ax4.text(-0.2, y_lora, 'LoRA\n(QLoRA)', ha='right', va='center', fontsize=10, fontweight='bold', color='#333')
ax4.text(cumx_full + 0.1, y_full, f'≈ {total_full} GB', va='center', fontsize=10, color='#e07b54', fontweight='bold')
ax4.text(cumx_lora + 0.1, y_lora, f'≈ {total_lora_est:.0f} GB', va='center', fontsize=10, color='#388e3c', fontweight='bold')

patches = [mpatches.Patch(color=c, label=cat.replace('\n', ' ')) for cat, c in zip(categories, colors_cat)]
ax4.legend(handles=patches, loc='lower right', fontsize=8.5, ncol=2)
ax4.text(5.5, 2.0, f'节省约 {100*(1-total_lora_est/total_full):.0f}% 显存（{total_full}GB → {total_lora_est:.0f}GB）',
         ha='center', fontsize=11, color='#388e3c', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor='#388e3c', alpha=0.8))

plt.tight_layout()
plt.show()

## 4.2.3 PEFT 生态

LoRA 是目前最主流的 PEFT 方法，但整个 PEFT 生态包含多种不同思路。理解它们的差异有助于针对不同场景选择合适的方法。

---

### Prompt Tuning（提示微调，2021）

**思想**：不修改任何模型参数，而是在输入序列前面**拼接一组可学习的"软提示"向量**（Soft Prompt），只训练这些向量。

```
传统 Prompt（硬提示，固定字符）：
  "Classify sentiment of: This movie is great."

Prompt Tuning（软提示，可学习向量）：
  [v₁, v₂, v₃, v₄, v₅] + "This movie is great."
  ↑ 这 5 个向量是可学习的连续向量，不对应任何词
```

**特点**：
- 极少的可训练参数（通常几百到几千个，即软提示的长度 × 模型维度）
- 随模型规模增大，效果越来越接近全量微调（但小模型上效果差）
- 可以为不同任务保存不同的软提示，模型本身只需一份

---

### Prefix Tuning（前缀微调，2021）

**思想**：比 Prompt Tuning 更深入——在每一层的 Attention 的 Key 和 Value 前面拼接可学习的前缀向量，而不是只在输入层。

```
标准 Self-Attention：
  K = [k₁, k₂, ..., kₙ],  V = [v₁, v₂, ..., vₙ]

Prefix Tuning：
  K = [p¹ₖ, p²ₖ, k₁, k₂, ..., kₙ]   ← 前缀 pₖ 是可学习的
  V = [p¹ᵥ, p²ᵥ, v₁, v₂, ..., vₙ]   ← 同理
```

每层都有独立的前缀参数，因此表达能力比 Prompt Tuning 强，但参数量也更多（层数 × 前缀长度 × d_model × 2）。

---

### Adapter（适配器，2019）

**思想**：在每个 Transformer 层的 Feed-Forward 网络后面插入一个小型"适配器"模块（两层全连接 + 残差），只训练适配器的参数。

```
FFN 输出
   │
   ▼
[Down-project: d → r]   ← r 很小，如 64
[非线性激活（ReLU）]
[Up-project: r → d]
   │ +（残差）
   ▼
继续传播
```

Adapter 是 LoRA 的前身，思路类似（低维瓶颈结构），但 LoRA 在推理时可以合并进原始权重，**没有推理延迟**；Adapter 则在推理时仍需执行额外的网络层，会带来少量延迟。

---

### IA³（Infused Adapter by Inhibiting and Amplifying Inner Activations）

**思想**：通过乘以可学习的缩放向量来调整激活值，而不是加法插入。参数量极少（每层只需 3 个向量），在低数据量场景下效果很好。

---

### 各方法对比

| 方法 | 位置 | 参数量 | 推理额外开销 | 适用场景 |
|------|------|--------|------------|----------|
| **Prompt Tuning** | 输入层 | 极少（<1万）| 无 | 超大模型、跨任务共享 |
| **Prefix Tuning** | 每层 K/V | 少（~几十万）| 微小（序列长度增加）| 生成任务 |
| **Adapter** | FFN 后 | 少（~几十万）| 有（额外层计算）| 通用 |
| **LoRA** | 权重矩阵旁路 | 少（~几百万）| 无（可合并）| 当前最主流 |
| **IA³** | 激活缩放 | 极少（~几千）| 无 | 低数据量任务 |
| **QLoRA** | LoRA + 4bit 量化 | 少（~几百万）| 无（可合并）| 单卡微调大模型 |

---

### HuggingFace PEFT 库

HuggingFace 提供了统一的 PEFT 库，3 行代码就能给任意模型加上 LoRA：

```python
from peft import get_peft_model, LoraConfig, TaskType

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # 秩
    lora_alpha=16,                # 缩放系数 α
    target_modules=['q_proj', 'v_proj'],  # 应用于哪些层
    lora_dropout=0.05,
)
model = get_peft_model(base_model, config)
model.print_trainable_parameters()
# trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
```

## 4.2.4 指令微调（Instruction Fine-tuning）

### 什么是指令微调？

预训练模型（Base Model）只会"续写"——给它一段文字，它就接着写。但用户真正需要的是能**理解并执行指令**的助手（"帮我总结这段话"、"把这段代码改成 Python 3"）。

**指令微调（SFT，Supervised Fine-tuning）**：用人工整理的"指令-回复"对话数据，对 Base Model 进行微调，让模型学会"听懂指令、给出有帮助的回复"。

```
Base Model（只会续写）          →  Chat Model（理解并执行指令）
"今天天气很好，我..."           →  用户："帮我写一首关于春天的诗"
 接着续写...                         模型：[有结构、有逻辑的诗歌]
```

---

### SFT 数据构造

**格式**：每个训练样本是一个对话（指令 + 理想回复）

```json
{
  "instruction": "把下面这段 Python 2 代码改成 Python 3",
  "input": "print 'Hello World'\nxrange(10)",
  "output": "print('Hello World')\nrange(10)"
}
```

**数据来源**：
1. **人工标注**：雇佣人类专家撰写高质量指令-回复对（成本高，质量好）
2. **Self-Instruct**：让强大的 LLM（如 GPT-4）生成指令和回复，再筛选（Stanford Alpaca、WizardLM）
3. **开源数据集**：FLAN（1800+ NLP 任务），ShareGPT（真实用户对话），Dolly（Databricks 发布的 15k 数据集）

---

### Chat Template（对话模板）

不同模型有各自的对话格式规范，必须严格遵守，否则模型不能正确区分"系统提示/用户/助手"。

**LLaMA-2 Chat 格式**：
```
<s>[INST] <<SYS>>
You are a helpful assistant.
<</SYS>>

帮我解释什么是梯度下降 [/INST] 梯度下降是... </s>
<s>[INST] 那学习率怎么选？ [/INST]
```

**ChatML 格式**（GPT-4、Qwen 等）：
```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
帮我解释什么是梯度下降<|im_end|>
<|im_start|>assistant
梯度下降是...
```

**重要**：SFT 训练时，损失函数只计算助手回复部分的 token（不计算用户输入和系统提示），这样模型学的是"如何生成好的回复"，而不是"如何生成用户说的话"。

---

### SFT → RLHF 的完整流程

```
阶段 1：SFT（监督微调）
  Base Model
      ↓ 用 "指令-回复" 数据微调
  SFT Model（能执行指令，但回复质量不稳定）

阶段 2：Reward Model 训练
  SFT Model 对同一问题生成多个回复
  人类标注员对这些回复排序（哪个最好？）
  训练 Reward Model 预测人类偏好分数

阶段 3：PPO 强化学习（RLHF）
  SFT Model
      ↓ 用 Reward Model 的分数作为奖励信号
      ↓ PPO 算法优化
  Chat Model（HHH: Helpful, Harmless, Honest）
```

**DPO（Direct Preference Optimization，2023）**：RLHF 的简化替代方案，直接用人类偏好数据优化，无需单独训练 Reward Model，实现更简单，效果接近。目前大量开源模型（如 Llama-3、Qwen2.5）采用 DPO 或其变体代替 RLHF。

---

### 本章小结

| 场景 | 推荐方案 |
|------|----------|
| 单卡 GPU 微调 7B 模型 | QLoRA（4bit + LoRA） |
| 多卡有限资源微调大模型 | LoRA（BF16） |
| 需要极少数据快速适配新任务 | IA³ 或 Prompt Tuning |
| 构建对话助手（开源基础模型）| SFT → DPO |
| 构建对齐、安全的商业模型 | SFT → RLHF |